# 03 — Modelagem
Mínimo de dois classificadores distintos. Um único modelo não atende ao requisito.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42
TARGET = "mau_pagador"

df = pd.read_csv("dados_processados.csv")

## 1. Split treino/teste
`stratify` preserva a proporção das classes nos dois conjuntos.

In [5]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Salvar bases de teste para o Notebook 04
X_test.to_csv('X_teste.csv', index=False)
y_test.to_csv('y_teste.csv', index=False)

## 2. Modelos candidatos
Usar Pipeline evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [3]:
modelos = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        # Random Forest não precisa de scaler, mas mantemos o formato do dicionário
        ("clf", RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
}

## 3. Validação cruzada
Avaliando a robustez dos modelos sem tocar na base de teste.

In [6]:
# Código idêntico ao sugerido pelo template da FIAP
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    print(nome, "F1-Score médio:", scores.mean().round(4))

Regressão Logística F1-Score médio: 0.0422
Random Forest F1-Score médio: 0.3044


## 4. Comparação
Leitura: qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?

**Conclusão:**
O modelo Random Forest obteve um F1-Score superior ao da Regressão Logística na validação cruzada. Por ser um modelo baseado em árvores de decisão, ele lida melhor com variáveis que não têm uma relação linear direta com o risco de crédito. O Random Forest será o modelo escolhido para avançar à etapa final de avaliação.

In [9]:
import joblib

# 5. Treinamento Final
modelo_escolhido = modelos["Random Forest"]
modelo_escolhido.fit(X_train, y_train)

joblib.dump(modelo_escolhido, 'modelo_rf.pkl')
print("Modelo Random Forest treinado e salvo!")

Modelo Random Forest treinado e salvo!
